# Metric Presentation and Visualization
## Necessary packages and functions call

- DDPM-TS: Interpretable Diffusion for Time Series Generation
- Metrics: 
    - discriminative_metrics
    - predictive_metrics
    - visualization

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

## Data Loading ETTh Morning Peak

Load original dataset and preprocess the loaded data.

In [2]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/morning_peak_etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_morning_peak_etth_milestone_1000_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2881, 24, 7)
fake shape is:  (20700, 24, 1)
ori shape is:  (20167, 24, 1)
fake shape is:  (20700, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [3]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Please use tf.global_variables instead.


training: 100%|██████████| 2000/2000 [00:56<00:00, 35.52it/s]


Iter 0:  0.00198314328210214 , 0.5480912245909767 , 0.447942488844819 



training: 100%|██████████| 2000/2000 [00:54<00:00, 36.65it/s]


Iter 1:  0.01152702032721864 , 0.5557759048091224 , 0.4672781358453148 



...

Iter 2:  0.0004957858205255627 , 0.4806643529995042 , 0.5203272186415469 



training: 100%|██████████| 2000/2000 [00:50<00:00, 40.00it/s]


Iter 3:  0.010659395141298988 , 0.5250371839365394 , 0.4962816063460585 



training: 100%|██████████| 2000/2000 [00:53<00:00, 37.31it/s]


Iter 4:  0.00198314328210214 , 0.5233019335647 , 0.4727317798710957 

etth:
Final Score:  0.005329697570649494 ± 0.0065872012815730505



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [4]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [02:03<00:00, 40.54it/s]


0  epoch:  0.16064100329310416 



training: 100%|██████████| 5000/5000 [02:00<00:00, 41.65it/s]


1  epoch:  0.1606334978875141 



training: 100%|██████████| 5000/5000 [01:57<00:00, 42.50it/s]


2  epoch:  0.16073196597746311 



training: 100%|██████████| 5000/5000 [01:51<00:00, 45.03it/s]


3  epoch:  0.16064525811717428 



training: 100%|██████████| 5000/5000 [01:55<00:00, 43.24it/s]


4  epoch:  0.16064123755219464 

sine:
Final Score:  0.16065859256549006 ± 5.120184441426329e-05



## Data Loading ETTh Evening Peak

Load original dataset and preprocess the loaded data.

In [5]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/evening_peak_etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_evening_peak_etth_milestone_1000_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2880, 24, 7)
fake shape is:  (20700, 24, 1)
ori shape is:  (20160, 24, 1)
fake shape is:  (20700, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [6]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


training: 100%|██████████| 2000/2000 [00:46<00:00, 43.19it/s]


Iter 0:  0.004588293650793662 , 0.4635416666666667 , 0.527281746031746 



training: 100%|██████████| 2000/2000 [00:48<00:00, 41.07it/s]


Iter 1:  0.006076388888888895 , 0.49702380952380953 , 0.4908234126984127 



training: 100%|██████████| 2000/2000 [00:50<00:00, 39.48it/s]


Iter 2:  0.011160714285714302 , 0.47197420634920634 , 0.5057043650793651 



training: 100%|██████████| 2000/2000 [00:47<00:00, 42.11it/s]


Iter 3:  0.00905257936507936 , 0.48933531746031744 , 0.49255952380952384 



training: 100%|██████████| 2000/2000 [00:47<00:00, 42.12it/s]


Iter 4:  0.001984126984126977 , 0.4937996031746032 , 0.5101686507936508 

etth:
Final Score:  0.006572420634920639 ± 0.004495735619243233



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [7]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [01:53<00:00, 43.97it/s]


0  epoch:  0.13088494388282254 



training: 100%|██████████| 5000/5000 [02:03<00:00, 40.34it/s]


1  epoch:  0.130867894700303 



training: 100%|██████████| 5000/5000 [01:53<00:00, 44.04it/s]


2  epoch:  0.1308733592965573 



training: 100%|██████████| 5000/5000 [02:02<00:00, 40.77it/s]


3  epoch:  0.13086736041161523 



training: 100%|██████████| 5000/5000 [02:18<00:00, 36.18it/s]


4  epoch:  0.13087717144098654 

sine:
Final Score:  0.1308741459464569 ± 9.028558640972522e-06



## Data Loading Energy Morning Peak

Load original dataset and preprocess the loaded data.

In [8]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/morning_peak_energy_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_morning_peak_energy_milestone_1000_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2580, 24, 28)
fake shape is:  (72900, 24, 1)
ori shape is:  (72240, 24, 1)
fake shape is:  (72900, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [9]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


training: 100%|██████████| 2000/2000 [01:07<00:00, 29.73it/s]


Iter 0:  0.005052602436323328 , 0.44345238095238093 , 0.5666528239202658 



training: 100%|██████████| 2000/2000 [01:05<00:00, 30.66it/s]


Iter 1:  0.009413067552602405 , 0.507890365448505 , 0.5109357696566998 



training: 100%|██████████| 2000/2000 [01:04<00:00, 30.85it/s]


Iter 2:  0.009101605758582543 , 0.5048449612403101 , 0.5133582502768549 



training: 100%|██████████| 2000/2000 [01:06<00:00, 30.23it/s]


Iter 3:  0.012769933554817259 , 0.13960409745293467 , 0.8859357696566998 



training: 100%|██████████| 2000/2000 [01:05<00:00, 30.47it/s]


Iter 4:  0.004879568106312293 , 0.5056755260243633 , 0.5040836101882613 

etth:
Final Score:  0.008243355481727565 ± 0.004122375871032591



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [10]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [02:32<00:00, 32.81it/s]


0  epoch:  0.20362009035420736 



training: 100%|██████████| 5000/5000 [02:31<00:00, 33.06it/s]


1  epoch:  0.2036137694834544 



training: 100%|██████████| 5000/5000 [02:31<00:00, 33.02it/s]


2  epoch:  0.2036569302799728 



training: 100%|██████████| 5000/5000 [02:31<00:00, 33.02it/s]


3  epoch:  0.20359617918647852 



training: 100%|██████████| 5000/5000 [02:31<00:00, 32.95it/s]


4  epoch:  0.2035866145720191 

sine:
Final Score:  0.2036147167752264 ± 3.369195667782235e-05



## Data Loading energy Evening Peak

Load original dataset and preprocess the loaded data.

In [11]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/evening_peak_energy_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_evening_peak_energy_milestone_1000_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2587, 24, 28)
fake shape is:  (72900, 24, 1)
ori shape is:  (72436, 24, 1)
fake shape is:  (72900, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [12]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


training: 100%|██████████| 2000/2000 [01:05<00:00, 30.62it/s]


Iter 0:  0.011975427940364436 , 0.5289204859193816 , 0.4950303699613473 



training: 100%|██████████| 2000/2000 [01:05<00:00, 30.39it/s]


Iter 1:  0.005142186637216994 , 0.5235367200441745 , 0.48674765323025954 



training: 100%|██████████| 2000/2000 [01:05<00:00, 30.44it/s]


Iter 2:  0.023191606847045798 , 0.547073440088349 , 0.4993097736057427 



training: 100%|██████████| 2000/2000 [01:06<00:00, 30.25it/s]


Iter 3:  0.021362506902263934 , 0.5277471010491441 , 0.5149779127553837 



training: 100%|██████████| 2000/2000 [00:59<00:00, 33.38it/s]


Iter 4:  0.005797901711761422 , 0.1129900607399227 , 0.8986057426836003 

etth:
Final Score:  0.013493926007730517 ± 0.010522063141507249



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [13]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [02:32<00:00, 32.85it/s]


0  epoch:  0.18617810259289036 



training: 100%|██████████| 5000/5000 [02:32<00:00, 32.82it/s]


1  epoch:  0.18623767903268526 



training: 100%|██████████| 5000/5000 [02:32<00:00, 32.79it/s]


2  epoch:  0.18619798352817743 



training: 100%|██████████| 5000/5000 [02:33<00:00, 32.62it/s]


3  epoch:  0.18633384922356172 



training: 100%|██████████| 5000/5000 [02:30<00:00, 33.24it/s]


4  epoch:  0.1864477091536495 

sine:
Final Score:  0.18627906470619288 ± 0.00013871472732484625

